#Objective

Same preprocessing for column values for Data preparation - 
1. Fixing the Binary values where missing is actually NO


In [32]:
import polars as pl
from pathlib import Path

In [33]:

df = pl.read_parquet(f'./data/interim/1.1-Hip-Reduced.parquet')

In [34]:

# For columns with exactly 2 unique values where one is 9, recode 9 → 0
cols_to_recode = [
    c for c in df.columns
    if set(df[c].drop_nulls().unique().to_list()) == {9, 0}
    or (
        len(df[c].drop_nulls().unique()) == 2
        and 9 in df[c].drop_nulls().unique().to_list()
    )
]

print(f"Columns to recode (2 unique values, one is 9): {cols_to_recode}")

df = df.with_columns([
    pl.when(pl.col(c) == 9).then(0).otherwise(pl.col(c)).alias(c)
    for c in cols_to_recode
])

print("Done. Value 9 replaced with 0 in the above columns.")


Columns to recode (2 unique values, one is 9): ['Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System', 'Liver Disease', 'Cancer', 'Depression', 'Arthritis']
Done. Value 9 replaced with 0 in the above columns.


In [36]:
# Recode 'Pre-Op Q Assisted By': 9 → 0, anything else → 1
df = df.with_columns(
    pl.when(pl.col("Pre-Op Q Assisted By") == 9)
      .then(0)
      .otherwise(1)
      .alias("Pre-Op Q Assisted By")
)

print("'Pre-Op Q Assisted By' recoded: 9 → 0, all other values → 1")
print(df["Pre-Op Q Assisted By"].value_counts())

#Post-Op Q Assisted By 

df = df.with_columns(
    pl.when(pl.col("Post-Op Q Assisted By") == 9)
      .then(0)
      .otherwise(1)
      .alias("Post-Op Q Assisted By")
)

print("'Post-Op Q Assisted By' recoded: 9 → 0, all other values → 1")
print(df["Post-Op Q Assisted By"].value_counts())


'Pre-Op Q Assisted By' recoded: 9 → 0, all other values → 1
shape: (1, 2)
┌──────────────────────┬────────┐
│ Pre-Op Q Assisted By ┆ count  │
│ ---                  ┆ ---    │
│ i32                  ┆ u32    │
╞══════════════════════╪════════╡
│ 1                    ┆ 124844 │
└──────────────────────┴────────┘
'Post-Op Q Assisted By' recoded: 9 → 0, all other values → 1
shape: (1, 2)
┌───────────────────────┬────────┐
│ Post-Op Q Assisted By ┆ count  │
│ ---                   ┆ ---    │
│ i32                   ┆ u32    │
╞═══════════════════════╪════════╡
│ 1                     ┆ 124844 │
└───────────────────────┴────────┘


In [37]:
# Replace common missing value indicators with null
# Numeric and string columns must be handled separately — is_in() requires matching types
numeric_dtypes = [pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64, pl.Float32, pl.Float64]
numeric_replace = [9, 999]
string_replace  = ["*", ""]

numeric_exprs = [
    pl.when(pl.col(c).is_in(numeric_replace)).then(None).otherwise(pl.col(c)).alias(c)
    for c in df.columns if df[c].dtype in numeric_dtypes
]
string_exprs = [
    pl.when(pl.col(c).is_in(string_replace)).then(None).otherwise(pl.col(c)).alias(c)
    for c in df.columns if df[c].dtype == pl.Utf8
]

df = df.with_columns(numeric_exprs + string_exprs)
print(df.null_count())

shape: (1, 82)
┌─────────────┬───────────┬────────────┬──────┬───┬────────────┬────────────┬────────────┬─────────┐
│ Provider    ┆ Procedure ┆ Revision   ┆ Year ┆ … ┆ Hip Replac ┆ Hip Replac ┆ Hip Replac ┆ CSVYear │
│ Code        ┆ ---       ┆ Flag       ┆ ---  ┆   ┆ ement      ┆ ement      ┆ ement OHS  ┆ ---     │
│ ---         ┆ u32       ┆ ---        ┆ u32  ┆   ┆ Post-Op Q  ┆ Post-Op Q  ┆ Post-Op Q  ┆ u32     │
│ u32         ┆           ┆ u32        ┆      ┆   ┆ Work       ┆ Scor…      ┆ …          ┆         │
│             ┆           ┆            ┆      ┆   ┆ ---        ┆ ---        ┆ ---        ┆         │
│             ┆           ┆            ┆      ┆   ┆ u32        ┆ u32        ┆ u32        ┆         │
╞═════════════╪═══════════╪════════════╪══════╪═══╪════════════╪════════════╪════════════╪═════════╡
│ 0           ┆ 0         ┆ 0          ┆ 0    ┆ … ┆ 962        ┆ 1627       ┆ 4038       ┆ 0       │
└─────────────┴───────────┴────────────┴──────┴───┴────────────┴────────────

In [38]:

# Column type summary: numeric vs categorical
numeric_dtypes = [pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64, pl.Float32, pl.Float64]

numeric_cols = [c for c in df.columns if df[c].dtype in numeric_dtypes]
cat_cols     = [c for c in df.columns if df[c].dtype in (pl.Utf8, pl.Categorical)]
other_cols   = [c for c in df.columns if c not in numeric_cols and c not in cat_cols]

print(f"Numeric columns     : {len(numeric_cols)}")
print(f"Categorical columns : {len(cat_cols)}")
print(f"  Names: {cat_cols}")
if other_cols:
    print(f"Other columns       : {len(other_cols)} — {[(c, df[c].dtype) for c in other_cols]}")


Numeric columns     : 75
Categorical columns : 7
  Names: ['Provider Code', 'Procedure', 'Year', 'Age Band', 'Hip Replacement EQ VAS Post-Op Q Predicted', 'Hip Replacement OHS Post-Op Q Predicted', 'CSVYear']


In [39]:

# Unique values for each categorical column (to plan encoding/transforms)
for c in cat_cols:
    unique_vals = df[c].drop_nulls().unique().cast(pl.Utf8).sort().to_list()
    print(f"\n{c!r}  (dtype={df[c].dtype}, {len(unique_vals)} unique values)")
    print(f"  {unique_vals}")



'Provider Code'  (dtype=Categorical, 292 unique values)
  ['ADP02', 'AHH', 'AVQ', 'NFH01', 'NN401', 'NN801', 'NQM01', 'NT202', 'NT204', 'NT205', 'NT206', 'NT209', 'NT210', 'NT211', 'NT212', 'NT213', 'NT214', 'NT215', 'NT218', 'NT219', 'NT224', 'NT225', 'NT226', 'NT229', 'NT230', 'NT233', 'NT235', 'NT237', 'NT238', 'NT239', 'NT241', 'NT242', 'NT244', 'NT245', 'NT301', 'NT302', 'NT304', 'NT305', 'NT308', 'NT309', 'NT30A', 'NT310', 'NT312', 'NT313', 'NT314', 'NT315', 'NT316', 'NT317', 'NT318', 'NT319', 'NT320', 'NT321', 'NT322', 'NT324', 'NT325', 'NT327', 'NT332', 'NT333', 'NT337', 'NT339', 'NT344', 'NT345', 'NT347', 'NT348', 'NT350', 'NT351', 'NT364', 'NT3X3', 'NT401', 'NT402', 'NT403', 'NT404', 'NT405', 'NT406', 'NT408', 'NT409', 'NT410', 'NT411', 'NT412', 'NT413', 'NT414', 'NT416', 'NT417', 'NT418', 'NT419', 'NT420', 'NT421', 'NT422', 'NT423', 'NT424', 'NT427', 'NT428', 'NT429', 'NT430', 'NT431', 'NT432', 'NT433', 'NT434', 'NT435', 'NT436', 'NT437', 'NT438', 'NT439', 'NT440', 'NT441',

In [40]:

# Label-encode selected categorical columns as integers, starting from 1 (0 is reserved for "No")
# Ordered: Year, CSVYear (chronological), Age Band (natural age order)
# Unordered: Procedure (alphabetical)

cols_to_encode = ["Procedure", "Year", "Age Band", "Provider Code"   ]  # --- IGNORE ---

age_band_order = sorted(df["Age Band"].drop_nulls().unique().cast(pl.Utf8).to_list())
# ↑ Replace with explicit list if alphabetical sort does not match the natural age order
# e.g. age_band_order = ["Under 45", "45 to 49", "50 to 54", "55 to 59", "60 to 64",
#                        "65 to 69", "70 to 74", "75 to 79", "80 to 84", "85 to 89", "90 and over"]

explicit_orders = {
    "Year":     sorted(df["Year"].drop_nulls().unique().cast(pl.Utf8).to_list()),
    "CSVYear":  sorted(df["CSVYear"].drop_nulls().unique().cast(pl.Utf8).to_list()),
    "Age Band": age_band_order,
}

encoding_maps = {}
for c in cols_to_encode:
    order = explicit_orders.get(c, sorted(df[c].drop_nulls().unique().cast(pl.Utf8).to_list()))
    encoding_maps[c] = {label: code for code, label in enumerate(order, start=1)}

# Apply encoding in-place (categorical → Int32)
df = df.with_columns([
    pl.col(c)
      .cast(pl.Utf8)
      .replace({k: str(v) for k, v in encoding_maps[c].items()})
      .cast(pl.Int32)
      .alias(c)
    for c in cols_to_encode
])

# Print mappings for reference
for c, mapping in encoding_maps.items():
    print(f"\n{c!r}")
    for label, code in mapping.items():
        print(f"  {code:>3} → {label!r}")



'Procedure'
    1 → 'Hip Replacement'

'Year'
    1 → '2016/17'
    2 → '2017/18'
    3 → '2018/19'

'Age Band'
    1 → '20 to 29'
    2 → '30 to 39'
    3 → '40 to 49'
    4 → '50 to 59'
    5 → '60 to 69'
    6 → '70 to 79'
    7 → '80 to 89'
    8 → '90 to 120'

'Provider Code'
    1 → 'ADP02'
    2 → 'AHH'
    3 → 'AVQ'
    4 → 'NFH01'
    5 → 'NN401'
    6 → 'NN801'
    7 → 'NQM01'
    8 → 'NT202'
    9 → 'NT204'
   10 → 'NT205'
   11 → 'NT206'
   12 → 'NT209'
   13 → 'NT210'
   14 → 'NT211'
   15 → 'NT212'
   16 → 'NT213'
   17 → 'NT214'
   18 → 'NT215'
   19 → 'NT218'
   20 → 'NT219'
   21 → 'NT224'
   22 → 'NT225'
   23 → 'NT226'
   24 → 'NT229'
   25 → 'NT230'
   26 → 'NT233'
   27 → 'NT235'
   28 → 'NT237'
   29 → 'NT238'
   30 → 'NT239'
   31 → 'NT241'
   32 → 'NT242'
   33 → 'NT244'
   34 → 'NT245'
   35 → 'NT301'
   36 → 'NT302'
   37 → 'NT304'
   38 → 'NT305'
   39 → 'NT308'
   40 → 'NT309'
   41 → 'NT30A'
   42 → 'NT310'
   43 → 'NT312'
   44 → 'NT313'
   45 → 'NT314'
 

In [41]:

# Drop columns that are not useful for modelling
cols_to_drop = [
                           # unique per provider, not a useful feature
    "Hip Replacement EQ VAS Post-Op Q Predicted",
    "Hip Replacement OHS Post-Op Q Predicted",
    "CSVYear",                                # redundant with Year after encoding
]

# Only drop columns that actually exist in the dataframe
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(cols_to_drop)

print(f"Dropped {len(cols_to_drop)} columns: {cols_to_drop}")
print(f"Remaining columns: {df.shape[1]}")


Dropped 3 columns: ['Hip Replacement EQ VAS Post-Op Q Predicted', 'Hip Replacement OHS Post-Op Q Predicted', 'CSVYear']
Remaining columns: 79


In [42]:

df.write_parquet(f'./data/interim/2.0-Hip-preprocessing.parquet', compression='gzip')